In [50]:
from dotenv import load_dotenv
load_dotenv()

from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
from pydantic import BaseModel
from langgraph.types import interrupt, Command
from IPython.display import Image, display
from typing import Literal
from langgraph.checkpoint.memory import InMemorySaver


class MailState(BaseModel):
    query: str = ""
    draft: str = ""
    human_feedback: str = ""
    final_response: str = ""


llm = ChatGroq(model="openai/gpt-oss-20b")

def DraftEmail(state: MailState) -> MailState:
    if state.human_feedback:
      
        prompt = f"""
        You are an expert assistant writing and refining emails.
        Original Request: {state.query}
        Current Draft: {state.draft}
        User Feedback/Changes Required: {state.human_feedback}
        
        Instructions: Rewrite the draft email to fully incorporate the user feedback. 
        If the feedback specifies new dates, completely replace the old dates. 
        Output ONLY the final revised email body without any conversational filler or multiple options.
        """
        res = llm.invoke(prompt)
        state.draft = res.content
    else:
        state.draft = llm.invoke(state.query).content
    
    return state
def HumanFeedBack(state: MailState) -> MailState:
    feedback = interrupt({
        "draft_email": state.draft,
        "question": "Do you want to continue or re-write the mail"
    })

    fb = (feedback or "").strip().lower()
    if fb in ("approved", "ok", "yes"):
        state.human_feedback = ""
        return state
    else:
        state.human_feedback = feedback
        return state


def Finalize_node(state: MailState) -> MailState:
    if state.human_feedback:
        state.final_response = state.human_feedback
        return state
    else:
        print("Email sent Successfully ...")
        state.final_response = "Email sent successfully"
        return state


def condition_routng(state: MailState) -> Literal["draft", "final"]:
    if state.human_feedback:
        return "draft"
    return "final"


graph_builder = StateGraph(MailState)

graph_builder.add_node("draft", DraftEmail)
graph_builder.add_node("human_fb", HumanFeedBack)
graph_builder.add_node("final", Finalize_node)

graph_builder.add_edge(START, "draft")
graph_builder.add_edge("draft", "human_fb")
graph_builder.add_conditional_edges("human_fb", condition_routng)
graph_builder.add_edge("final", END)

graph = graph_builder.compile(checkpointer=InMemorySaver())


config = {"configurable": {"thread_id": "1"}}
res = graph.invoke({
    "query": "I want to send a email for Medical Leave Request, dates will be 26 oct to 30 oct, so create a simple mail."
}, config=config)

print("Draft to review:", res)


user_input = "I want to change the leave dates, and new dates are 5th november to 7th november"

while True:
   
    res = graph.invoke(Command(resume=user_input), config=config)
    
    
    if isinstance(res, dict) and "draft_email" in res:
        print("\n--- New Draft Generated ---")
        print(res["draft_email"])
        
        
        user_input = input("Enter feedback or type 'approved': ")
        if user_input.strip().lower() in ("approved", "ok", "yes"):
            user_input = "approved"
    else:
        print("\nFinal Result:", res)
        break

Draft to review: {'query': 'I want to send a email for Medical Leave Request, dates will be 26 oct to 30 oct, so create a simple mail.', 'draft': '**Subject:** Medical Leave Request (26\u202fOct\u202f–\u202f30\u202fOct)\n\n**Email Body:**\n\nDear [Supervisor’s Name],\n\nI am writing to request medical leave from **26\u202fOctober to 30\u202fOctober**. I have attached the doctor’s note for your reference.\n\nI will ensure all my current tasks are up‑to‑date before I leave and will be reachable for any urgent questions during this period.\n\nThank you for your understanding.\n\nBest regards,\n\n[Your Full Name]  \n[Your Position]  \n[Your Contact Information]', 'human_feedback': '', 'final_response': '', '__interrupt__': [Interrupt(value={'draft_email': '**Subject:** Medical Leave Request (26\u202fOct\u202f–\u202f30\u202fOct)\n\n**Email Body:**\n\nDear [Supervisor’s Name],\n\nI am writing to request medical leave from **26\u202fOctober to 30\u202fOctober**. I have attached the doctor’s n